# 10.5 DAG & Job Execution

## Learning Objectives

By the end of this lesson, you will understand:

- DAG Scheduler
- Jobs
- Stages
- Tasks
- Narrow Transformations
- Wide Transformations

These concepts explain how Spark executes your code internally.

---

# Recap From Previous Lesson

We learned:

- Transformations are lazy
- Actions trigger execution
- Spark builds a DAG

Today we answer:

What happens after an Action is called?

<img src="../assets/Module_10/10_05_01.png" alt="Apache Spark execution flow diagram. User Code → DAG → Job → Stages → Tasks → Executors. Clean educational infographic, beginner friendly, white background, professional training style." width="75%">

---

# Big Picture

When an Action is triggered:

Spark does NOT directly execute code.

Instead it creates:

DAG
 ↓
Job
 ↓
Stages
 ↓
Tasks

Then execution begins.

---

# DAG SECTION

---

# What is a DAG?

DAG stands for:

**D**irected
**A**cyclic
**G**raph

A DAG represents all transformations and their dependencies.

Think of it as Spark's execution blueprint.

---

# Example

Read CSV
 ↓
Filter
 ↓
Select
 ↓
GroupBy
 ↓
Count

This sequence becomes a DAG.

<img src="../assets/Module_10/10_05_02.png" alt="Apache Spark DAG visualization. Read CSV node connected to Filter node, then Select node, then GroupBy node, then Count action. Arrows showing dependencies. Educational architecture diagram, white background." width="75%">

In [ ]:
df = (
    spark.read.csv("employees.csv", header=True)
         .filter("salary > 50000")
         .select("department")
         .groupBy("department")
         .count()
)

---

# Has Spark Executed Anything Yet?

No.

Spark only built a DAG.

Execution starts only when an Action is called.

---

# DAG SCHEDULER

---

# What is the DAG Scheduler?

The DAG Scheduler is responsible for:

- Analyzing the DAG
- Breaking DAG into Stages
- Scheduling execution

It acts as the project manager of execution.

---

# DAG Scheduler Responsibilities

1. Build Execution Plan
2. Identify Shuffle Boundaries
3. Create Stages
4. Submit Tasks to Executors

Without the DAG Scheduler, Spark cannot execute jobs efficiently.

<img src="../assets/Module_10/10_05_03.png" alt="Apache Spark DAG Scheduler illustration. DAG Scheduler in center analyzing a DAG and splitting it into multiple stages before sending tasks to executors. Educational technical diagram, white background." width="75%">

---

# JOBS

---

# What is a Job?

A Job is created whenever an Action is triggered.

Examples:

`count()`

`show()`

`collect()`

`write()`

Each Action creates a new Job.

In [ ]:
df.count()

---

# What Happened?

Spark created:

**Job #1**

because `count()` is an Action.

In [ ]:
df.show()

---

# Another Job

`show()` is another Action.

Spark creates:

**Job #2**

Each Action generates a separate Job.

<img src="../assets/Module_10/10_05_04.png" alt="Apache Spark illustration showing two actions (count and show) creating two separate Spark jobs. Educational infographic explaining actions and jobs, white background." width="75%">

---

# STAGES

---

# What is a Stage?

A Stage is a group of Tasks that can execute together.

Jobs are divided into Stages.

Stages are separated by Shuffle operations.

---

# Why Do Stages Exist?

Spark tries to execute as much work as possible together.

But when data must move across partitions:

A new Stage is required.

<img src="../assets/Module_10/10_05_05.png" alt="Apache Spark stage visualization. Stage 1 contains Read and Filter operations. Shuffle boundary shown. Stage 2 contains GroupBy and Count operations. Clear educational diagram." width="75%">

---

# Example

Read CSV
 ↓
Filter
 ↓
**Shuffle**
 ↓
GroupBy
 ↓
Count

**Stage 1:**
Read + Filter

**Stage 2:**
GroupBy + Count

---

# TASKS

---

# What is a Task?

A Task is the smallest unit of work in Spark.

Tasks are sent to Executors.

Executors perform the actual computation.

---

# Relationship

Job
 ↓
Stages
 ↓
Tasks

A single Job may contain:

- Multiple Stages
- Hundreds of Tasks

<img src="../assets/Module_10/10_05_06.png" alt="Apache Spark hierarchy diagram showing Job at top, multiple Stages beneath it, and multiple Tasks under each Stage. Clean educational architecture infographic, white background." width="75%">

---

# Example

Suppose:

Dataset = 4 Partitions

Stage = 1

Spark creates:

Task 1
Task 2
Task 3
Task 4

One task per partition.

<img src="../assets/Module_10/10_05_07.png" alt="Apache Spark partitions and tasks diagram. Dataset split into four partitions, generating four tasks executed by executors in parallel. Beginner friendly educational visualization." width="75%">

---

# NARROW VS WIDE

---

# Narrow vs Wide Transformations

This is one of the most important Spark concepts.

It determines:

- Performance
- Shuffle Cost
- Number of Stages

---

# Narrow Transformation

A child partition depends on only one parent partition.

Examples:

`filter()`

`select()`

`withColumn()`

No shuffle required.

<img src="../assets/Module_10/10_05_08.png" alt="Apache Spark narrow transformation diagram. Partition 1 maps to Partition 1, Partition 2 maps to Partition 2, Partition 3 maps to Partition 3. No data movement. Educational white background." width="75%">

In [ ]:
df.filter("salary > 50000")

---

# Why Narrow Transformations Are Fast

No data movement.

No network transfer.

No shuffle.

Therefore:

Fast execution.

---

# Wide Transformation

A child partition depends on multiple parent partitions.

Examples:

`groupBy()`

`join()`

`distinct()`

Data movement is required.

<img src="../assets/Module_10/10_05_09.png" alt="Apache Spark wide transformation diagram. Multiple parent partitions sending data across network to multiple child partitions during shuffle. Educational Spark shuffle visualization, white background." width="75%">

In [ ]:
df.groupBy("department").count()

---

# Why Wide Transformations Are Expensive

Wide transformations trigger:

- Shuffle
- Network Traffic
- Disk Spill Possibility
- Additional Stages

This increases execution time.

---

# Comparison

| Feature | Narrow | Wide |
|----------|----------|----------|
| Shuffle | No | Yes |
| Speed | Faster | Slower |
| Network Traffic | Minimal | High |
| Additional Stage | Usually No | Usually Yes |

<img src="../assets/Module_10/10_05_10.png" alt="Side-by-side comparison infographic of Spark Narrow vs Wide Transformations. Narrow shows no shuffle, Wide shows network shuffle. Educational beginner friendly diagram." width="75%">

---

# COMPLETE EXECUTION FLOW

---

# Complete Spark Execution Flow

User Code
 ↓
DAG
 ↓
Action
 ↓
Job
 ↓
Stages
 ↓
Tasks
 ↓
Executors
 ↓
Results

<img src="../assets/Module_10/10_05_11.png" alt="Complete Apache Spark execution pipeline diagram. User Code → DAG → Job → Stages → Tasks → Executors → Results. Modern educational infographic, white background." width="75%">

---

# Real World Analogy

Imagine building a house.

**Entire House** = Job

**Foundation, Walls, Roof** = Stages

**Individual Workers** = Tasks

**Project Manager** = DAG Scheduler

This is exactly how Spark organizes work.

---

# Key Takeaways

✓ DAG represents execution flow

✓ DAG Scheduler creates stages

✓ Actions create Jobs

✓ Jobs contain Stages

✓ Stages contain Tasks

✓ Narrow Transformations are faster

✓ Wide Transformations cause Shuffle

**Next Lesson:**

10.6 Spark Partitions Fundamentals